<a href="https://colab.research.google.com/github/Vasista-G/GoogleCollab_Python/blob/main/PandasLvL3/PandasLvL3Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PANDAS LEVEL 3 - PRACTICE QUESTIONS**

In [111]:
import pandas as pd

emp = pd.read_csv("employee.csv")
dept = pd.read_csv("department.csv")
bonus = pd.read_csv("bonus.csv")
sales = pd.read_csv("sales.csv")

emp.head()


,employee_id,name,age,department,city,salary,experience_years,joining_date,performance_rating
0,101,Aarav Sharma,25.0,IT,Hyderabad,55000.0,2.0,2022-06-15,4.0
1,102,Priya Reddy,29.0,HR,Hyderabad,62000.0,5.0,2019-03-20,5.0
2,103,Rahul Kumar,32.0,Finance,Bangalore,75000.0,8.0,2016-08-10,4.0
3,104,Ananya Singh,27.0,IT,Pune,68000.0,4.0,2020-01-12,3.0
4,105,Vikram Patel,35.0,Sales,Mumbai,82000.0,10.0,2014-11-05,5.0


In [ ]:
dept.head()

,department,manager,location,budget
0,IT,Raj Malhotra,Hyderabad,5000000
1,HR,Priya Reddy,Hyderabad,2500000
2,Finance,Rahul Kumar,Bangalore,4000000
3,Sales,Vikram Patel,Mumbai,4500000
4,Marketing,Neha Verma,Delhi,3000000


In [ ]:
#2.	Merge employees and departments using the department column.

m = pd.merge(emp, dept, on = "department", how = "inner")

m[["name", "department","manager", "location"]]


In [ ]:
#5.	Find employees who work in the same city as their department location.

m.loc[m["city"] == m["location"], ["name", "city", "location"]]

In [ ]:
#7.	After merging, find the average salary for each manager.

m.groupby("manager")["salary"].mean()

,salary
manager,
Neha Verma,56500.000000
Priya Reddy,58500.000000
Rahul Kumar,81222.222222
Raj Malhotra,74000.000000
Vikram Patel,77800.000000


In [ ]:
#8.	Find the total number of employees managed by each manager.

m.groupby("manager")["employee_id"].count()



,employee_id
manager,
Neha Verma,10
Priya Reddy,8
Rahul Kumar,10
Raj Malhotra,12
Vikram Patel,10


In [ ]:
#9.	Find the highest-paid employee under each manager.

result = (
    m.loc[
        m.groupby("manager")["salary"].idxmax(),
        ["salary", "name", "manager"]
    ]
)

result


,salary,name,manager
43,63000.0,Kavya Pillai,Neha Verma
1,62000.0,Priya Reddy,Priya Reddy
8,95000.0,Rohan Gupta,Rahul Kumar
40,110000.0,Raj Malhotra,Raj Malhotra
14,85000.0,Manish Yadav,Vikram Patel


In [ ]:
#10.	Find the average salary of employees in each department along with the department budget.

m.groupby("department")[["salary", "budget"]].mean()


,salary,budget
department,,
Finance,81222.222222,4000000.0
HR,58500.000000,2500000.0
IT,74000.000000,5000000.0
Marketing,56500.000000,3000000.0
Sales,77800.000000,4500000.0


# **SECTION 2 - LEFT, RIGHT AND INNER MERGE**

In [47]:
ebm = pd.merge(emp, bonus, on = "employee_id", how = "inner")


In [ ]:
#Find employees who do not have a bonus record.

ebm.loc[~ ebm["employee_id"].isin(bonus["employee_id"]), ["employee_id", "bonus"]]

In [ ]:
#Find employees who have received a bonus.

emp[ebm["bonus"].notna()]

In [42]:
#Compare the number of rows produced by:  INNER JOIN , LEFT JOIN

print(len(ebm))

ebm = pd.merge(emp, bonus, on = "employee_id", how = "inner")

print(len(ebm))

50
19


In [46]:
#Find employees whose bonus is greater than 8,000.

ebm.loc[(ebm["bonus"] > 8000) & (ebm["salary"] > 70000), ["employee_id", "bonus"]]

,employee_id,bonus
4,105,10000.0
6,107,9000.0
8,109,12000.0
14,115,11000.0
16,117,13000.0
18,119,8500.0
40,141,15000.0
42,143,10000.0
48,149,9000.0


# **SECTION 3 - MERGE + CALCULATED COLUMNS**

In [ ]:
'''
Merge employees with bonuses.
Create a new column: total_compensation
Formula:
salary + bonus

Create: bonus_percentage Formula: bonus / salary * 100

'''

ebm["total_compensation"] = ebm["salary"] + ebm["bonus"]

ebm["bonus_percentage"] = ebm["bonus"] / ebm["salary"] * 100

ebm[["total_compensation", "bonus_percentage"]]



In [77]:
# 21 Find the employee with the highest total compensation.

ebm.sort_values(by = "total_compensation", ascending = False, ignore_index= True)[["total_compensation", "employee_id"]].nlargest(1, "total_compensation")

,total_compensation,employee_id
0,125000.0,141


In [110]:
#25 Find the department with the highest total bonus.

ebm.groupby("department", as_index = False)["bonus"].sum().nlargest(1, "bonus")



,department,bonus
2,IT,61000


# **SECTION 4 - SALES DATA**

In [141]:
#Create a new column: sales_amount Formula: quantity * unit_price
#32 Find the top 5 sales transactions.

sales["sales_amount"] = sales["quantity"] * sales["unit_price"]

sales.sort_values(by = "sales_amount", ascending = False)[["product", "category","sales_amount"]].nlargest(5, "sales_amount")

,product,category,sales_amount
16,Laptop,Electronics,210000
15,Phone,Electronics,144000
1,Phone,Electronics,125000
11,Laptop,Electronics,124000
7,Laptop,Electronics,116000


In [154]:
#33 Find total sales by product.

sales.groupby("product")["sales_amount"].idxmax()

,sales_amount
product,
Keyboard,18
Laptop,16
Monitor,17
Mouse,19
Phone,15


In [148]:
sales.loc[sales.groupby("product")["sales_amount"].idxmax(), ["product", "category", "sales_amount"]]

,product,category,sales_amount
18,Keyboard,Accessories,27600
16,Laptop,Electronics,210000
17,Monitor,Electronics,105000
19,Mouse,Accessories,27500
15,Phone,Electronics,144000


# **SECTION 5 - SALES + EMPLOYEE MERGE**

In [167]:
sem = pd.merge(emp, sales, on = "employee_id", how = "inner")

#43 Find the employee who generated the highest total sales.

sem.groupby("employee_id")["sales_amount"].sum().nlargest(1)

,sales_amount
employee_id,
105,249000
